# 03 — Profiling & Silver Cleaning.


O Data Profiling vai ajudar-nos a entender a qualidade dos dados (nulos, duplicados e distribuição). A Silver Cleaning aplicará as regras de negócio para corrigir esses problemas e ajustar os esquemas (schema casting).


### 3.1 Setup inicial
Nesta fase vamos realizar os imports, carregar as tabelas previamente guardadas em formato Delta e definir os caminhos base de cada uma delas.


In [0]:
# Setup inicial
# imports necessários, definição de caminhos base necessários e lista com as tabelas a trabalhar
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Caminhos Delta necessários para correr este notebook de forma independente
bronze_delta_path = "/Volumes/main/default/faers_data/delta/bronze"
silver_delta_path = "/Volumes/main/default/faers_data/delta/silver"

tables = ["demo", "drug", "reac", "outc"]


In [0]:
# Primeiro carregamos as tabelas Delta da camada Bronze
bronze_dfs = {}
for table in tables:
    bronze_dfs[table] = spark.read.format("delta").load(f"{bronze_delta_path}/{table}/")


In [0]:
# visualização das tabelas
for table in tables:
    print(f"\nTabela: {table}")
    display(bronze_dfs[table].limit(5))



### 3.2 Profiling inicial
Após verificarmos que as tabelas bronze foram corretamente carregadas vamos começar a fazer um profiling inicial.

Nesta fase queremos perceber a estrutura e qualidade dos dados antes de definir regras de limpeza.


### 3.2.1 Contagens e schemas

In [0]:
# Faz-se uma contagem do nº de registos e de colunas de cada tabela na fase bronze.
# Neste caso, todas as tabelas foram carregadas com um schema que define todas as colunas como string.
# Ainda assim, é importante realizar uma última verificação.
for table, df in bronze_dfs.items():
    print(f"Tabela {table.upper()} apresenta {bronze_dfs[table].count():,} registos.")
    print(f"Tabela {table.upper()} apresenta {len(bronze_dfs[table].columns)} colunas.")
    print(f"\nSchema da tabela {table.upper()}:")
    df.printSchema()


### 3.2.2 Verificação de nulos


In [0]:
import pyspark.sql.functions as F

# Iterar sobre cada tabela guardada no dicionário bronze_dfs
for table_name, df in bronze_dfs.items():
    print(f"--- Percentagem de Nulos para a Tabela: {table_name.upper()} ---")
    
    total_rows = df.count()
    
    # Prevenção de erro caso alguma tabela não tenha registos (divisão por zero)
    if total_rows == 0:
        print("A tabela está vazia.\n")
        continue

    null_analysis = df.select(
        *[
            F.round(
                F.sum(
                    F.when(
                        (F.col(c).isNull()) | (F.trim(F.col(c)) == ""),1).otherwise(0)) / total_rows * 100,1
                ).alias(c)
            for c in df.columns
            ]
    )
    
    display(null_analysis)


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

for table_name, df in bronze_dfs.items():
    print(f"--- Análise Detalhada de Nulos: {table_name.upper()} ---")
    
    total_rows = df.count()
    
    if total_rows == 0:
        print("A tabela está vazia.\n")
        continue

    # 1. Calcula a contagem de nulos para todas as colunas
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
    null_counts_row = df.select(*null_exprs).collect()[0]
    
    # 2. Constrói a lista APENAS com colunas que têm nulos
    summary_data = []
    for c in df.columns:
        n_nulls = null_counts_row[c]
        
        # Só adiciona à lista se tiver pelo menos 1 nulo
        if n_nulls > 0:
            pct_nulls = round((n_nulls / total_rows) * 100, 2)
            summary_data.append((c, n_nulls, pct_nulls))
            
    # 3. Se a lista estiver vazia (zero nulos na tabela inteira), avisa e avança para a próxima tabela
    if not summary_data:
        print("🎉 Não existem colunas com valores nulos nesta tabela!\n")
        continue
        
    # 4. Define o esquema para o novo DataFrame
    schema = StructType([
        StructField("Nome_Coluna", StringType(), True),
        StructField("Qtd_Nulos", LongType(), True),
        StructField("%_Nulos", DoubleType(), True)
    ])
    
    # 5. Cria o DataFrame, ordena de forma decrescente e exibe
    summary_df = spark.createDataFrame(summary_data, schema)
    summary_df = summary_df.orderBy(F.col("Qtd_Nulos").desc())
    
    display(summary_df)


### 3.3.1 Tratamento de Nulos

O tratamento de valores nulos numa arquitetura Medallion, especialmente ao lidar com dados de saúde do mundo real provenientes de sistemas de reporte voluntário como o FAERS, exige um equilíbrio rigoroso. A simples eliminação de todas as linhas com nulos introduziria um forte viés na amostra, ocultando eventos adversos valiosos.

A estratégia aplicada nesta transição para a camada Silver baseia-se nos seguintes princípios de qualidade de dados:

1. **Integridade Relacional (Drop de Chaves Nulas):** Registos cujas chaves identificadoras (`primaryid` ou `caseid`) sejam nulas são eliminados. Sem estes identificadores, torna-se impossível garantir os *joins* entre tabelas (relacionar corretamente um paciente ao seu medicamento e à respetiva reação adversa), tornando o registo órfão e inútil para o pipeline.
2. **Preservação do Contexto Clínico:** Colunas numéricas contínuas (como `age` ou `wt`) e de datas mantêm os seus valores nulos originais. Em estudos de farmacovigilância, a imputação estatística (usar médias ou medianas) em demografia clínica é perigosa, pois pode mascarar a realidade fenotípica do paciente e falsificar a análise final.
3. **Padronização de Categóricas (FillNA com 'UNK'):** Variáveis categóricas essenciais recebem o valor explícito `"UNK"` (Unknown) ou `"U"`. Isto garante que os modelos analíticos e os *dashboards* downstream contabilizem a "falta de informação" como uma categoria válida e auditável, em vez de lidar com *missing values* imprevisíveis do motor Spark.


In [0]:
# Criação de um novo dicionário para armazenar os DataFrames limpos (Silver)
silver_dfs = {}

for table_name, df in bronze_dfs.items():
    
    # 1. Eliminar registos que não tenham as chaves identificadoras fundamentais
    df_clean = df.dropna(subset=["primaryid", "caseid"])
    
    # 2. Tratamento específico por tabela: Preenchimento de nulos em categóricas
    if table_name == "demo":
        df_clean = df_clean.fillna({
            "sex": "UNK", 
            # "age_grp": "UNK",  <-- Mantido comentado para evitar o erro de coluna não encontrada
            "occp_cod": "UNK", 
            "reporter_country": "UNK",
            "e_sub": "U"
        })
    elif table_name == "drug":
        df_clean = df_clean.fillna({
            "role_cod": "UNK", 
            "route": "UNK",
            "dechal": "U", 
            "rechal": "U",
            "dose_freq": "UNK"
        })
    elif table_name == "reac":
        df_clean = df_clean.fillna({
            "drug_rec_act": "UNK"
        })
        
    silver_dfs[table_name] = df_clean
    
    # Validação do impacto das transformações
    registos_iniciais = df.count()
    registos_finais = df_clean.count()
    registos_removidos = registos_iniciais - registos_finais
    
    print(f"--- Tabela {table_name.upper()} ---")
    print(f"Total de registos ANTES do tratamento: {registos_iniciais:,}")
    print(f"Registos removidos (falta de primaryid/caseid): {registos_removidos:,}")
    print(f"Total de registos APÓS tratamento: {registos_finais:,}\n")

# Atualizar o dicionário principal para as próximas etapas (ex: duplicados) usarem os dados já limpos de nulos críticos
bronze_dfs = silver_dfs


### 3.4 Verificação de duplicados
A análise de duplicados será usada para definir a estratégia de limpeza na camada Silver.
- 
- Na camada Silver serão removidos duplicados exatos e serão mantidos os identificadores necessários para preservar relações entre tabelas. *A deduplicação por chave lógica será aplicada com cuidado, uma vez que algumas tabelas FAERS podem conter múltiplos registos válidos por caso, medicamento, reação ou desfecho.*


In [0]:
for table, df in bronze_dfs.items():
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicate_rows = total_rows - distinct_rows
    
    print(f"{table.upper()}")
    print(f"Total: {total_rows}")
    print(f"Distintos: {distinct_rows}")
    print(f"Duplicados exatos: {duplicate_rows}")
    print("-" * 40)


### 3.4.1 Tratamento de Duplicados

O profiling revelou a presença de duplicados exatos, com particular incidência na tabela `REAC` (mais de 100 mil registos). No contexto do FAERS, isto ocorre frequentemente devido a redundâncias no preenchimento do formulário original ou em submissões de acompanhamento (*follow-ups*) onde os mesmos sintomas são recarregados.

**Estratégia de Limpeza:**
Uma vez que são duplicados exatos (todas as colunas contêm os mesmos valores), estes registos não acrescentam qualquer contexto clínico novo. Pelo contrário, mantê-los causaria enviesamento e dupla contagem (*double-counting*) na fase de modelação ou na criação de dashboards. 

Aplica-se a função `dropDuplicates()` a todas as tabelas para garantir a integridade da camada Silver, mantendo apenas registos únicos para cada combinação de caso, medicamento e reação.


In [0]:
print("=== Tratamento De Duplicados Exatos===\n")

# Dicionário temporário para guardar os DataFrames sem duplicados
dedup_dfs = {}

for table_name, df in bronze_dfs.items():
    # 1. Contagem inicial (antes da remoção)
    total_rows_antes = df.count()
    
    # 2. Remover duplicados exatos (avalia todas as colunas por defeito)
    df_dedup = df.dropDuplicates()
    
    # 3. Contagem final e cálculo da diferença
    total_rows_depois = df_dedup.count()
    duplicados_removidos = total_rows_antes - total_rows_depois
    
    # 4. Guardar o DataFrame limpo no novo dicionário
    dedup_dfs[table_name] = df_dedup
    
    # Mostrar resultados
    print(f"--- Tabela: {table_name.upper()} ---")
    print(f"Total antes: {total_rows_antes:,}")
    print(f"Duplicados removidos: {duplicados_removidos:,}")
    print(f"Total depois: {total_rows_depois:,}\n")

# Atualizar o dicionário principal com os dados agora sem nulos críticos e sem duplicados
bronze_dfs = dedup_dfs


### 3.5 Schema Casting (Conversão de Tipos de Dados)

Na camada Bronze, todas as colunas foram ingeridas temporariamente como `string` para garantir a fidelidade aos ficheiros originais. Agora, na camada Silver, é necessário atribuir os tipos de dados semânticos corretos (Datas, Números Inteiros e Decimais).

**Principais Transformações:**
1. **Datas:** Os ficheiros FAERS utilizam o formato `AAAAMMDD`. Colunas como `event_dt` ou `fda_dt` serão convertidas para `DateType`.*
2. **Métricas Clínicas e Doses:** Colunas quantitativas como `age` (idade), `wt` (peso) e `dose_amt` (quantidade da dose) serão convertidas para `DoubleType` para permitir agregações matemáticas (médias, distribuições) na camada Gold.
3. As variáveis categóricas e identificadores (como `primaryid`, `pt`, `outc_cod`) mantêm-se como `string`.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

print("=== SCHEMA CASTING E GRAVAÇÃO SILVER ===\n")

# Dicionário para guardar as tabelas finais da camada Silver
silver_final_dfs = {}

for table_name, df in bronze_dfs.items():
    df_cast = df
    
    # Transformações específicas para a tabela DEMO
    if table_name == "demo":
        df_cast = df_cast.withColumn("event_dt", F.to_date(F.col("event_dt"), "yyyyMMdd")) \
                         .withColumn("mfr_dt", F.to_date(F.col("mfr_dt"), "yyyyMMdd")) \
                         .withColumn("init_fda_dt", F.to_date(F.col("init_fda_dt"), "yyyyMMdd")) \
                         .withColumn("fda_dt", F.to_date(F.col("fda_dt"), "yyyyMMdd")) \
                         .withColumn("rept_dt", F.to_date(F.col("rept_dt"), "yyyyMMdd")) \
                         .withColumn("age", F.col("age").cast(DoubleType())) \
                         .withColumn("wt", F.col("wt").cast(DoubleType()))
                         
    # Transformações específicas para a tabela DRUG
    elif table_name == "drug":
        df_cast = df_cast.withColumn("exp_dt", F.to_date(F.col("exp_dt"), "yyyyMMdd")) \
                         .withColumn("dose_amt", F.col("dose_amt").cast(DoubleType())) \
                         .withColumn("cum_dose_chr", F.col("cum_dose_chr").cast(DoubleType()))
                         
    # As tabelas REAC e OUTC contêm apenas identificadores e códigos em texto, 
    # pelo que não necessitam de casting numérico/temporal.
    
    silver_final_dfs[table_name] = df_cast
    print(f"Schema atualizado para a tabela: {table_name.upper()}")

print("\n--- A Iniciar Gravação na Camada Silver ---")

# Gravação em formato Delta
for table_name, df in silver_final_dfs.items():
    output_path = f"{silver_delta_path}/{table_name}"
    
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .option("overwriteSchema", "true")
          .save(output_path)
    )
    print(f"✅ Tabela {table_name.upper()} gravada com sucesso em: {output_path}")

# Atualiza os dados em memória para verificação se necessário
silver_dfs = silver_final_dfs


### 3.6 Validação Final da Camada Silver (Data Quality Check)

Antes de darmos a camada Silver como concluída, realizamos uma auditoria final diretamente nos ficheiros Delta que foram gravados no Unity Catalog/Volume. 

Esta validação garante que:
1. O motor Spark consegue ler as tabelas gravadas sem corrupção.
2. O **Schema Casting** foi persistido corretamente (verificando os tipos `date` e `double`).
3. Visualizamos uma amostra real dos dados já limpos de nulos críticos, sem duplicados e com a tipagem correta, prontos para alimentar a camada Gold.


In [0]:
print("=== AUDITORIA E VALIDAÇÃO DA CAMADA SILVER (DELTA) ===\n")

for table in tables:
    silver_path = f"{silver_delta_path}/{table}"
    
    print(f" Matriz de Validação para a tabela: {table.upper()}")
    
    # Ler diretamente do caminho Delta gravado
    df_silver = spark.read.format("delta").load(silver_path)
    
    # 1. Contagem total de linhas salvas
    total_rows = df_silver.count()
    print(f"   -> Total de registos persistidos: {total_rows:,}")
    print(f"   -> Total de colunas: {len(df_silver.columns)}")
    
    # 2. Print do Schema para validar visualmente o Casting
    print("   -> Estrutura do Schema:")
    df_silver.printSchema()
    
    # 3. Mostrar uma amostra rápida dos dados limpos
    print(f"   -> Amostra dos primeiros 3 registos de {table.upper()}:")
    display(df_silver.limit(3))
    
    print("-" * 80)


#### 3.4.2 Duplicados por chaves lógicas
Nas tabelas FAERS, algumas colunas funcionam como identificadores importantes. Para este projeto, devemos analisar sobretudo :
| Tabela | Colunas relevantes                   |
| ------ | ------------------------------------ |
| DEMO   | `primaryid`, `caseid`, `caseversion` |
| DRUG   | `primaryid`, `caseid`, `drug_seq`    |
| REAC   | `primaryid`, `caseid`, `pt`          |
| OUTC   | `primaryid`, `caseid`, `outc_cod`    |


In [0]:
demo_df = bronze_dfs["demo"]

display(
     demo_df
     .groupBy("primaryid", "caseid", "caseversion")
     .count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
     .limit(5)
 )


In [0]:
drug_df = bronze_dfs["drug"]
 
display(
     drug_df
     .groupBy("primaryid", "caseid", "drug_seq")
     .count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
     .limit(5)
)


In [0]:
reac_df = bronze_dfs["reac"]
 
display(
     reac_df
     .groupBy("primaryid", "caseid", "pt")
     .count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
     .limit(5)
 )


In [0]:
outc_df = bronze_dfs["outc"]
 
display(
     outc_df
     .groupBy("primaryid", "caseid", "outc_cod")
     .count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
     .limit(5)
 )
